# 第3节 · 在免费 GPU 上部署 vLLM 并压测

在 **Kaggle**（Settings → Accelerator → GPU T4/P100）或 **Colab**（代码执行程序 → 更改运行时类型 → T4 GPU）里从上到下运行。

做三件事：
1. 用 vLLM 起一个 **OpenAI 兼容** 的推理服务（Qwen2.5-1.5B-Instruct，开源免登录）。
2. 用仓库里的 `serving/benchmark.py` 压测不同并发下的 **吞吐 / TTFT / TPOT**。
3. **量化与显存**：换成 7B 的 AWQ(int4) 模型，对比显存占用，并用 `serving/kvcache_calc.py` 现场估算。

把每一步的数字抄进 `docs/serving_report.md`。

## 0. 环境检查 + 安装 vLLM

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# vLLM 会一并装好匹配的 torch；首次安装约 2-3 分钟
!pip -q install "vllm==0.11.0"

## 1. 后台启动 vLLM 的 OpenAI 服务

`vllm serve <model>` 会在 `:8000` 上开一个和 OpenAI 一样的 `/v1` 接口。我们用子进程后台起它，再轮询 `/v1/models` 等它就绪。

In [ ]:
import subprocess, time, urllib.request, os

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 开源、免登录、单卡够跑
BASE = "http://localhost:8000/v1"

def start_vllm(model, extra=None):
    cmd = ["vllm", "serve", model,
           "--max-model-len", "4096",
           "--gpu-memory-utilization", "0.9",
           "--disable-log-requests"] + (extra or [])
    log = open("vllm.log", "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    return proc

def wait_ready(timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(BASE + "/models", timeout=5) as r:
                if r.status == 200:
                    print("服务就绪，用时 %.0fs" % (time.time() - t0)); return True
        except Exception:
            time.sleep(5)
    print("启动超时，看 vllm.log"); return False

proc = start_vllm(MODEL)
wait_ready()

In [ ]:
# 冒烟测试：发一条，确认能流式返回
!python -c "import urllib.request,json; \
req=urllib.request.Request('http://localhost:8000/v1/chat/completions', \
data=json.dumps({'model':'Qwen/Qwen2.5-1.5B-Instruct','messages':[{'role':'user','content':'一句话解释安全库存'}],'max_tokens':64}).encode(), \
headers={'Content-Type':'application/json'}); \
print(json.load(urllib.request.urlopen(req))['choices'][0]['message']['content'])"

## 2. 压测：吞吐 / TTFT / TPOT 随并发变化

拉仓库，直接用 `serving/benchmark.py`（纯标准库）。把输出表格抄进报告。

In [ ]:
![ -d supplypilot ] || git clone -q https://github.com/kevin0008-tutorial/supplypilot.git
%cd supplypilot
!git switch lession_3 2>/dev/null; cd ..
!python -m serving.benchmark --base-url http://localhost:8000/v1 \
    --model Qwen/Qwen2.5-1.5B-Instruct --concurrency 1,2,4,8,16,32 --requests 64 --max-tokens 128

## 3. 量化与显存：7B 模型怎么塞进免费 GPU

FP16 的 7B 权重要 ~14GB，免费 T4(16GB) 很勉强。换成 **AWQ(int4)** 量化版，权重掉到 ~5.5GB，省出来的显存留给 KV Cache = 更高并发。

先关掉 1.5B 的服务，再起 7B-AWQ，对比 `nvidia-smi` 显存。

In [ ]:
proc.terminate(); time.sleep(10)
proc = start_vllm("Qwen/Qwen2.5-7B-Instruct-AWQ", extra=["--quantization", "awq_marlin"])
wait_ready()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
!python -m serving.benchmark --base-url http://localhost:8000/v1 \
    --model Qwen/Qwen2.5-7B-Instruct-AWQ --concurrency 1,4,8,16 --requests 64 --max-tokens 128

## 4. 显存估算（离线，不用 GPU）：把算式对一遍实测

In [ ]:
!python -m serving.kvcache_calc --model qwen2.5-7b --seq 4096 --vram 16 --weight-dtype int4
# 把这里算出的权重/每-token-KV/可并发条数，和上面 nvidia-smi、压测的并发上限对照

In [ ]:
proc.terminate()   # 收尾，释放 GPU